# Chapter 1 — Strategy Pattern
## *Welcome to Design Patterns: Intro to Design Patterns*

**Intent:** Define a family of algorithms, encapsulate each one, and make them interchangeable.  
Strategy lets the algorithm vary independently from clients that use it.

### OO Design Principles introduced
1. **Encapsulate what varies** — separate the parts that change from those that stay the same.
2. **Program to an interface, not an implementation.**
3. **Favor composition over inheritance.**

### Book context
The classic example is a `Duck` simulation app.  
- Bad design: `fly()` and `quack()` hardcoded in the base class → breaks when you add `RubberDuck`.  
- Good design: extract `FlyBehavior` and `QuackBehavior` into separate strategy classes, then **compose** them into each duck at runtime.

## Python building blocks: `ABC` and `@abstractmethod`

Before diving into the pattern, here are two Python tools used throughout this notebook.

### `ABC` — Abstract Base Class

`ABC` (from the `abc` module) is a helper class that marks a class as **abstract** — meaning it exists purely to define a contract, and you cannot instantiate it directly.

```python
from abc import ABC

class FlyBehavior(ABC):  # abstract — cannot do FlyBehavior()
    pass
```

If you try `FlyBehavior()` you get: `TypeError: Can't instantiate abstract class`.  
Its only job is to be subclassed.

---

### `@abstractmethod` — enforced interface

Decorating a method with `@abstractmethod` tells Python: *"any concrete subclass **must** override this method."*  
If a subclass forgets, Python raises a `TypeError` at instantiation time — catching the mistake early.

```python
from abc import ABC, abstractmethod

class FlyBehavior(ABC):
    @abstractmethod
    def fly(self) -> str: ...   # no body needed; subclasses must provide one
```

Together, `ABC` + `@abstractmethod` replace what other languages call **interfaces** or **abstract classes** — they let you program to an abstraction rather than a concrete implementation (Design Principle #2 above).

In [ ]:
# ── Strategy interfaces (program to an abstraction) ─────────────────────────
from abc import ABC, abstractmethod

class FlyBehavior(ABC):
    @abstractmethod
    def fly(self) -> str: ...

class QuackBehavior(ABC):
    @abstractmethod
    def quack(self) -> str: ...

In [ ]:
# ── Concrete strategies ──────────────────────────────────────────────────────
class FlyWithWings(FlyBehavior):
    def fly(self): return "I'm flying with wings!"

class FlyNoWay(FlyBehavior):
    def fly(self): return "I can't fly."

class FlyRocketPowered(FlyBehavior):
    def fly(self): return "I'm flying with a rocket!"


class Quack(QuackBehavior):
    def quack(self): return "Quack!"

class Squeak(QuackBehavior):
    def quack(self): return "Squeak!"

class MuteQuack(QuackBehavior):
    def quack(self): return "<silence>"

In [ ]:
# ── Context class: Duck ──────────────────────────────────────────────────────
class Duck:
    def __init__(self, fly_behavior: FlyBehavior, quack_behavior: QuackBehavior):
        self._fly_behavior = fly_behavior
        self._quack_behavior = quack_behavior

    # Swap strategies at runtime — key Strategy power
    def set_fly_behavior(self, fb: FlyBehavior):
        self._fly_behavior = fb

    def set_quack_behavior(self, qb: QuackBehavior):
        self._quack_behavior = qb

    def perform_fly(self):
        print(self._fly_behavior.fly())

    def perform_quack(self):
        print(self._quack_behavior.quack())

    def swim(self):
        print("All ducks float, even decoys!")


class MallardDuck(Duck):
    def __init__(self):
        super().__init__(FlyWithWings(), Quack())

    def display(self): print("I'm a real Mallard duck.")


class RubberDuck(Duck):
    def __init__(self):
        super().__init__(FlyNoWay(), Squeak())

    def display(self): print("I'm a rubber duck.")


class ModelDuck(Duck):
    def __init__(self):
        super().__init__(FlyNoWay(), Quack())

    def display(self): print("I'm a model duck.")

In [ ]:
# ── Demo ─────────────────────────────────────────────────────────────────────
print("=== Mallard ===")
mallard = MallardDuck()
mallard.display()
mallard.perform_fly()
mallard.perform_quack()

print("\n=== Rubber ===")
rubber = RubberDuck()
rubber.display()
rubber.perform_fly()
rubber.perform_quack()

print("\n=== Model duck gets a rocket at runtime ===")
model = ModelDuck()
model.display()
model.perform_fly()
model.set_fly_behavior(FlyRocketPowered())  # swap at runtime!
model.perform_fly()

## Real-world analogy — payment methods
An e-commerce checkout uses Strategy so you can swap `CreditCard`, `PayPal`, or `Crypto` without touching the `Order` class.

### Without Strategy — conditional branching

The naive approach hardcodes the payment logic inside `checkout()` using `if/elif`. Every time you add a new payment method, you must open and modify `ShoppingCart` — violating the **Open/Closed Principle** (open for extension, closed for modification).

In [ ]:
class ShoppingCartNaive:
    def __init__(self):
        self._items: list[tuple[str, float]] = []

    def add(self, name: str, price: float):
        self._items.append((name, price))

    def checkout(self, payment_type: str):
        total = sum(p for _, p in self._items)

        # Every new payment method = another elif here
        if payment_type == "credit_card":
            print(f"Paid ${total:.2f} with Credit Card.")
        elif payment_type == "paypal":
            print(f"Paid ${total:.2f} via PayPal.")
        elif payment_type == "crypto":
            print(f"Paid ${total:.2f} in Bitcoin.")
        else:
            raise ValueError(f"Unknown payment type: {payment_type}")


cart = ShoppingCartNaive()
cart.add("Book", 29.99)
cart.add("Keyboard", 89.99)

cart.checkout("credit_card")
cart.checkout("paypal")
cart.checkout("crypto")
# Adding "bank_transfer" later means editing this class

### With Strategy — no branching, swap at runtime

Each payment method becomes its own class. `ShoppingCart` never needs to change when you add a new one — you just pass a different strategy object in.

In [23]:
class PaymentStrategy(ABC):
    @abstractmethod
    def pay(self, amount: float) -> str: ...

class CreditCard(PaymentStrategy):
    def pay(self, amount): return f"Paid ${amount:.2f} with Credit Card."

class PayPal(PaymentStrategy):
    def pay(self, amount): return f"Paid ${amount:.2f} via PayPal."

class Crypto(PaymentStrategy):
    def pay(self, amount): return f"Paid ${amount:.2f} in Bitcoin."


class ShoppingCart:
    def __init__(self):
        self._items: list[tuple[str, float]] = []

    def add(self, name: str, price: float):
        self._items.append((name, price))

    def checkout(self, strategy: PaymentStrategy):
        total = sum(p for _, p in self._items)
        print(strategy.pay(total))


cart = ShoppingCart()
cart.add("Book", 29.99)
cart.add("Keyboard", 89.99)

cart.checkout(CreditCard())
cart.checkout(PayPal())
cart.checkout(Crypto())

Paid $119.98 with Credit Card.
Paid $119.98 via PayPal.
Paid $119.98 in Bitcoin.


## Interview cheat-sheet

| Question | Answer |
|---|---|
| What problem does Strategy solve? | Eliminates conditional branching when selecting algorithms; lets you swap them at runtime. |
| Strategy vs inheritance? | Inheritance shares code top-down; Strategy composes behavior objects, so changes are isolated. |
| When NOT to use? | Only one algorithm ever needed, or strategies share too much state with context. |
| Python built-in example? | `sorted(key=...)` — the `key` function is the strategy. |

**Pattern family:** Behavioral